In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

contact_email = os.getenv("WIKIPEDIA_CONTACT_EMAIL")

if not contact_email:
    raise ValueError("WIKIPEDIA_CONTACT_EMAIL is missing from .env")

headers = {
    "User-Agent": f"SweetPotatoRAG/1.0 (academic research; contact: {contact_email})"
}

In [ ]:
import requests

url = "https://en.wikipedia.org/w/api.php"

params = {
    "action": "query",
    "prop": "extracts",
    "titles": "Sweet potato",
    "explaintext": 1,
    "format": "json"
}

response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=30
)

response.raise_for_status()

data = response.json()

In [ ]:
pages = data["query"]["pages"]

page = next(iter(pages.values()))

print("Article title:", page["title"])
print("Text length:", len(page.get("extract", "")))
print("\nFirst 500 characters:")
print(page.get("extract", "")[:500])

In [ ]:
metadata_params = {
    "action": "query",
    "prop": "revisions|info",
    "titles": "Sweet potato",
    "rvprop": "ids|timestamp",
    "inprop": "url",
    "format": "json"
}

metadata_response = requests.get(
    url,
    params=metadata_params,
    headers=headers,
    timeout=30
)

metadata_response.raise_for_status()

metadata_data = metadata_response.json()
metadata_page = next(iter(metadata_data["query"]["pages"].values()))

print("Title:", metadata_page["title"])
print("URL:", metadata_page["fullurl"])
print("Revision ID:", metadata_page["revisions"][0]["revid"])
print("Last modified:", metadata_page["revisions"][0]["timestamp"])

In [ ]:
revision_id = metadata_page["revisions"][0]["revid"]

revision_params = {
    "action": "query",
    "prop": "extracts",
    "revids": revision_id,
    "explaintext": 1,
    "format": "json"
}

revision_response = requests.get(
    url,
    params=revision_params,
    headers=headers,
    timeout=30
)

revision_response.raise_for_status()

revision_data = revision_response.json()
revision_page = next(iter(revision_data["query"]["pages"].values()))

article_text = revision_page.get("extract", "")

print("Characters retrieved:", len(article_text))
print("Revision ID:", revision_id)

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

# Create folders for our corpus
output_dir = Path("corpus/new_files/wikipedia")
output_dir.mkdir(parents=True, exist_ok=True)

# Combine article text and metadata
article = {
    "title": revision_page["title"],
    "source": metadata_page["fullurl"],
    "revision_id": revision_id,
    "last_modified": metadata_page["revisions"][0]["timestamp"],
    "retrieved_at": datetime.now(timezone.utc).isoformat(),
    "license": "CC BY-SA 4.0",
    "text": article_text
}

# Save as JSON
output_path = output_dir / "sweet_potato.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(article, f, indent=2, ensure_ascii=False)

print("Saved:", output_path)

In [ ]:
# This function:
# - Accepts a search phrase.
# - Queries Wikipedia's search API.
# - Returns up to 10 article titles and page IDs.
# - Does not download or save any articles yet.

def search_wikipedia(query, limit=10):
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "srlimit": limit,
        "format": "json"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    results = response.json()["query"]["search"]

    return [
        {
            "title": item["title"],
            "pageid": item["pageid"]
        }
        for item in results
    ]

In [ ]:
results = search_wikipedia("sweet potato cultivation")

for article in results:
    print(article["title"], "-", article["pageid"])

In [ ]:
search_queries = [
    "sweet potato",
    "sweet potato cultivation",
    "sweet potato diseases",
    "sweet potato nutrition",
    "sweet potato breeding",
    "sweet potato storage",
    "Ipomoea batatas"
]

candidate_articles = {}

for query in search_queries:
    results = search_wikipedia(query, limit=10)

    for article in results:
        candidate_articles[article["pageid"]] = article["title"]

print("Unique candidate articles:", len(candidate_articles))

In [ ]:
for pageid, title in sorted(
    candidate_articles.items(),
    key=lambda item: item[1].lower()
):
    print(f"{pageid}: {title}")

In [ ]:
selected_titles = [
    "Batata",
    "Ceratocystis fimbriata",
    "Fried sweet potato",
    "Imo Shochu",
    "Ipomoea",
    "Kankoro mochi",
    "Kumara",
    "List of sweet potato cultivars",
    "List of sweet potato diseases",
    "List of sweet potato dishes",
    "Maria Andrade (scientist)",
    "Rhizopus soft rot",
    "Roasted sweet potato",
    "Robert Mwanga",
    "Sweet potato",
    "Sweet potato cultivation in Polynesia",
    "Sweet potato feathery mottle virus",
    "Sweet potato latent virus",
    "Sweet potato leaf curl virus",
    "Sweet potato pie",
    "Sweet potato soup",
    "Sweet potato storage",
    "Sweet potato yellow dwarf virus",
    "Agriculture in Papua New Guinea",
    "Bactericera cockerelli",
    "Root vegetable",
    "Selective breeding",
    "Tuber"
]

selected_articles = {
    pageid: title
    for pageid, title in candidate_articles.items()
    if title in selected_titles
}

print("Selected articles:", len(selected_articles))

In [ ]:
import re
import time

def download_wikipedia_article(pageid, title):
    # Get the latest revision ID and timestamp
    params = {
        "action": "query",
        "prop": "revisions|info",
        "pageids": pageid,
        "rvprop": "ids|timestamp",
        "inprop": "url",
        "format": "json"
    }

    response = requests.get(
        url, params=params, headers=headers, timeout=30
    )
    response.raise_for_status()

    page = next(iter(response.json()["query"]["pages"].values()))

    revision = page["revisions"][0]
    revision_id = revision["revid"]

    # Retrieve text from that exact revision
    text_params = {
        "action": "query",
        "prop": "extracts",
        "revids": revision_id,
        "explaintext": 1,
        "format": "json"
    }

    text_response = requests.get(
        url, params=text_params, headers=headers, timeout=30
    )
    text_response.raise_for_status()

    text_page = next(
        iter(text_response.json()["query"]["pages"].values())
    )

    article_text = text_page.get("extract", "")

    if not article_text.strip():
        print(f"Skipped (empty): {title}")
        return None

    # Save content and metadata
    article = {
        "title": page["title"],
        "pageid": int(pageid),
        "source": page["fullurl"],
        "revision_id": revision_id,
        "last_modified": revision["timestamp"],
        "retrieved_at": datetime.now(timezone.utc).isoformat(),
        "license": "CC BY-SA 4.0",
        "text": article_text
    }

    filename = re.sub(
        r"[^a-z0-9]+", "_", title.lower()
    ).strip("_") + ".json"

    output_path = output_dir / filename

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(article, f, indent=2, ensure_ascii=False)

    return output_path

In [ ]:
test_pageid, test_title = next(iter(selected_articles.items()))

saved_path = download_wikipedia_article(test_pageid, test_title)

print("Saved:", saved_path)

In [ ]:
downloaded = []
failed = []

for pageid, title in selected_articles.items():
    try:
        saved_path = download_wikipedia_article(pageid, title)

        if saved_path is not None:
            downloaded.append(str(saved_path))
            print(f"Saved: {title}")
        else:
            failed.append(title)

    except Exception as e:
        failed.append(title)
        print(f"Failed: {title} — {e}")

    time.sleep(1)  # Pause between requests

print("\nDownload complete!")
print("Successful:", len(downloaded))
print("Failed:", len(failed))
print("Failed articles:", failed)

In [ ]:
import pandas as pd

article_stats = []

for filepath in output_dir.glob("*.json"):
    with open(filepath, "r", encoding="utf-8") as f:
        article = json.load(f)

    article_stats.append({
        "title": article["title"],
        "characters": len(article["text"]),
        "revision_id": article["revision_id"],
        "filename": filepath.name
    })

stats_df = pd.DataFrame(article_stats)

print("Total documents:", len(stats_df))
print("Total characters:", stats_df["characters"].sum())

display(stats_df.sort_values("characters"))

In [ ]:
import hashlib

content_hashes = {}

for filepath in output_dir.glob("*.json"):
    with open(filepath, "r", encoding="utf-8") as f:
        article = json.load(f)

    text = article["text"].strip()
    content_hash = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    content_hashes.setdefault(content_hash, []).append(
        article["title"]
    )

duplicates = [
    titles
    for titles in content_hashes.values()
    if len(titles) > 1
]

print("Unique documents:", len(content_hashes))
print("Exact duplicates:", duplicates)

In [ ]:
# Remove from our selected article dictionary
selected_articles = {
    pageid: title
    for pageid, title in selected_articles.items()
    if title != "Selective breeding"
}

# Remove the downloaded file
filepath = output_dir / "selective_breeding.json"

if filepath.exists():
    filepath.unlink()
    print("Removed:", filepath.name)

print("Selected articles:", len(selected_articles))
print("Wikipedia files:", len(list(output_dir.glob("*.json"))))

In [6]:
# here is where the corpus building starts

In [7]:
from pathlib import Path

corpus_dir = Path("corpus")

wikipedia_files = list((corpus_dir / "new_files" / "wikipedia").glob("*.json"))
proquest_files = list((corpus_dir / "new_files" / "proquest").glob("*.pdf"))
original_files = list((corpus_dir / "original_files" / "original_pdfs").glob("*.pdf"))

print("Wikipedia:", len(wikipedia_files))
print("ProQuest:", len(proquest_files))
print("Original PDFs:", len(original_files))

total = (
    len(wikipedia_files)
    + len(proquest_files)
    + len(original_files)
)

print("Total documents:", total)

Wikipedia: 27
ProQuest: 44
Original PDFs: 7
Total documents: 78


In [8]:
import hashlib

pdf_files = proquest_files + original_files

pdf_hashes = {}

for filepath in pdf_files:
    file_hash = hashlib.sha256(
        filepath.read_bytes()
    ).hexdigest()

    pdf_hashes.setdefault(file_hash, []).append(str(filepath))

duplicates = [
    paths
    for paths in pdf_hashes.values()
    if len(paths) > 1
]

print("Total PDFs:", len(pdf_files))
print("Unique PDF files:", len(pdf_hashes))
print("Exact duplicates:", len(duplicates))

for group in duplicates:
    print("\nDuplicate group:")
    for path in group:
        print(" ", path)

Total PDFs: 51
Unique PDF files: 51
Exact duplicates: 0


In [9]:
from pypdf import PdfReader

pdf_results = []

for filepath in pdf_files:
    reader = PdfReader(filepath)

    text = "".join(
        page.extract_text() or ""
        for page in reader.pages
    )

    pdf_results.append({
        "filename": filepath.name,
        "pages": len(reader.pages),
        "characters": len(text.strip())
    })

pdf_df = pd.DataFrame(pdf_results)

print("PDFs checked:", len(pdf_df))
print("PDFs with no extractable text:",
      (pdf_df["characters"] == 0).sum())

display(pdf_df.sort_values("characters").head(10))

PDFs checked: 51
PDFs with no extractable text: 0


,filename,pages,characters
49,rag_sp6.pdf,6,12255
48,rag_sp5.pdf,3,16500
28,Node_Position_Determines_Sweet.pdf,9,23036
38,Productivity_enhancement_in_sw.pdf,6,24316
42,The_productivity_and_economic_.pdf,9,26729
9,Development_of_a_transferrable.pdf,8,34440
44,rag_sp1.pdf,8,41021
18,Evaluation_of_Tuber_Yield_Perf.pdf,9,41575
35,Phytochemical_Screening_and_Bi.pdf,12,41786
11,Effects_of_drying_on_the_bioac.pdf,11,42441


In [10]:
for filepath in pdf_files:
    reader = PdfReader(filepath)
    title = reader.metadata.title if reader.metadata else None

    print(f"{filepath.name}: {title}")

Advances_in_Molecular_Techniqu.pdf: Advances in Molecular Techniques for Detecting Sweet Potato (Ipomoea batatas (L.) Lam) Viruses: A Comprehensive Review
Advances_in_semiochemicals_of_.pdf: Advances in semiochemicals of the sweet potato weevil,
Biochemical_Responses_of_Ataca.pdf: Biochemical Responses of Atacama and Blesbok Sweet Potato (Ipomoea batatas L.) Cultivars to Early Drought Stress
Biofortification_of_Sweetpotat.pdf: Biofortification of Sweetpotato (Ipomoea batatas [L.] Lam.) in Cuba
Biotic_Stress_Resistance_in_Sw.pdf: Biotic Stress Resistance in Sweet Potato: Mechanisms, Perspectives, and Sustainable Production Strategies
Can_Hyperspectral_Reflectance_.pdf: Can Hyperspectral Reflectance Thresholds Achieve Spatial Partitioning of Sweet Potato Leaf Deformation Types on UAV Multispectral Imagery?
Changes_in_Chlorophyll_iai.pdf: Changes in Chlorophyll a Fluorescence in Ipomoea batatas (Convolvulaceae) Genotypes Under Attack by Bedellia somnulentella (Lepidoptera: Bedelliidae)
Co

In [11]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

pdf_token_counts = []

for filepath in pdf_files:
    reader = PdfReader(filepath)

    text = "\n".join(
        page.extract_text() or ""
        for page in reader.pages
    )

    pdf_token_counts.append({
        "filename": filepath.name,
        "tokens": len(encoding.encode(text))
    })

pdf_tokens = pd.DataFrame(pdf_token_counts)

print("PDF documents:", len(pdf_tokens))
print("Total PDF tokens:", pdf_tokens["tokens"].sum())

PDF documents: 51
Total PDF tokens: 907774


In [13]:
import json

In [14]:
manifest = []

for filepath in wikipedia_files:
    with open(filepath, "r", encoding="utf-8") as f:
        article = json.load(f)

    manifest.append({
        "document_id": f"wiki_{article['pageid']}",
        "title": article["title"],
        "source_type": "wikipedia",
        "source_url": article["source"],
        "file_path": filepath.as_posix(),
        "format": "json",
        "revision_id": article["revision_id"],
        "license": article["license"]
    })

print("Manifest entries:", len(manifest))
print(json.dumps(manifest[0], indent=2))

Manifest entries: 27
{
  "document_id": "wiki_31685001",
  "title": "Agriculture in Papua New Guinea",
  "source_type": "wikipedia",
  "source_url": "https://en.wikipedia.org/wiki/Agriculture_in_Papua_New_Guinea",
  "file_path": "corpus/new_files/wikipedia/agriculture_in_papua_new_guinea.json",
  "format": "json",
  "revision_id": 1376404528,
  "license": "CC BY-SA 4.0"
}


In [15]:
for filepath in proquest_files:
    reader = PdfReader(filepath)

    title = (
        reader.metadata.title
        if reader.metadata and reader.metadata.title
        else filepath.stem
    )

    manifest.append({
        "document_id": f"proquest_{filepath.stem}",
        "title": title,
        "source_type": "proquest",
        "source_url": None,
        "file_path": filepath.as_posix(),
        "format": "pdf",
        "revision_id": None,
        "license": "Unknown - institutional subscription",
    })

print("Manifest entries:", len(manifest))
print(json.dumps(manifest[-1], indent=2))

Manifest entries: 71
{
  "document_id": "proquest_Use_of_Microbial_and_Enzymatic",
  "title": "Use of Microbial and Enzymatic Additives on the Nutritional Quality, Fermentation Profile, and In Vitro Digestibility of Mixed Silages of Amaranth and Sweet Potato Vines",
  "source_type": "proquest",
  "source_url": null,
  "file_path": "corpus/new_files/proquest/Use_of_Microbial_and_Enzymatic.pdf",
  "format": "pdf",
  "revision_id": null,
  "license": "Unknown - institutional subscription"
}


In [16]:
for filepath in original_files:
    reader = PdfReader(filepath)

    title = (
        reader.metadata.title
        if reader.metadata and reader.metadata.title
        else filepath.stem
    )

    manifest.append({
        "document_id": f"original_{filepath.stem}",
        "title": title,
        "source_type": "original_pdf",
        "source_url": None,
        "file_path": filepath.as_posix(),
        "format": "pdf",
        "revision_id": None,
        "license": "Unknown - needs verification"
    })

print("Manifest entries:", len(manifest))
print(json.dumps(manifest[-1], indent=2))

Manifest entries: 78
{
  "document_id": "original_rag_sp7",
  "title": "Genetic Diversity and Population Structure of the USDA Sweetpotato (Ipomoea batatas) Germplasm Collections Using GBSpoly",
  "source_type": "original_pdf",
  "source_url": null,
  "file_path": "corpus/original_files/original_pdfs/rag_sp7.pdf",
  "format": "pdf",
  "revision_id": null,
  "license": "Unknown - needs verification"
}


In [17]:
from pathlib import Path

html_files = list((corpus_dir / "original_files" / "original_websites").rglob("*.htm"))
html_files += list((corpus_dir / "original_files" / "original_websites").rglob("*.html"))

csv_files = list((corpus_dir / "original_files").glob("*.csv"))

print("HTML files:")
for filepath in html_files:
    print(filepath)

print("\nCSV files:")
for filepath in csv_files:
    print(filepath)

HTML files:
corpus\original_files\original_websites\sp_wb3.htm
corpus\original_files\original_websites\sp_wb4.htm
corpus\original_files\original_websites\sp_wb5.htm
corpus\original_files\original_websites\sp_wp1.htm
corpus\original_files\original_websites\sp_wp2.htm

CSV files:
corpus\original_files\sp_table2.csv
corpus\original_files\sp_table3.csv


In [18]:
# Use only the copies inside corpus/
html_files = sorted(
    (corpus_dir / "original_files" / "original_websites").glob("*.htm")
)

csv_files = sorted(
    (corpus_dir / "original_files").glob("*.csv")
)

for filepath in html_files + csv_files:
    manifest.append({
        "document_id": f"original_{filepath.stem}",
        "title": filepath.stem,
        "source_type": "original_website" if filepath.suffix == ".htm" else "original_csv",
        "source_url": None,
        "file_path": filepath.as_posix(),
        "format": filepath.suffix.lstrip("."),
        "revision_id": None,
        "license": "Unknown - needs verification"
    })

print("HTML files added:", len(html_files))
print("CSV files added:", len(csv_files))
print("Total manifest entries:", len(manifest))

HTML files added: 5
CSV files added: 2
Total manifest entries: 85


In [19]:
from collections import Counter

# Check for duplicate document IDs
document_ids = [doc["document_id"] for doc in manifest]

duplicates = [
    doc_id
    for doc_id, count in Counter(document_ids).items()
    if count > 1
]

# Check that every referenced file exists
missing_files = [
    doc["file_path"]
    for doc in manifest
    if not Path(doc["file_path"]).is_file()
]

# Check for missing required metadata
required_fields = [
    "document_id",
    "title",
    "source_type",
    "file_path",
    "format",
    "license"
]

incomplete = [
    doc["document_id"]
    for doc in manifest
    if any(not doc.get(field) for field in required_fields)
]

print("Total entries:", len(manifest))
print("Duplicate IDs:", duplicates)
print("Missing files:", missing_files)
print("Incomplete entries:", incomplete)

Total entries: 85
Duplicate IDs: []
Missing files: []
Incomplete entries: []


In [20]:
manifest_path = Path("corpus/manifest.jsonl")

with open(manifest_path, "w", encoding="utf-8") as f:
    for document in manifest:
        f.write(json.dumps(document, ensure_ascii=False) + "\n")

print("Saved:", manifest_path)
print("Documents:", len(manifest))

Saved: corpus\manifest.jsonl
Documents: 85
